# EJERCICIOS DE HOLD OUT

![logo itq](img/ITQimagen.png)

**Nombre:** Bryan Inga
**Fecha:** 16/11/2026
**Link:** <a href="https://github.com/israelmena018-cmd/fundamentosDemachine67456387648.git">Repositorio en GitHub</a>

---

## ¿Qué es Hold Out (Mantener fuera)?

Es una técnica de validación que **divide los datos en subconjuntos separados**: uno para entrenar el modelo y otro(s) para evaluarlo con datos que nunca vio.

$$D = D_{train} \cup D_{val} \cup D_{test}, \qquad D_{train} \cap D_{val} \cap D_{test} = \emptyset$$

En este ejercicio se aplica **dos veces** una partición 80 % - 20 %:

| Partición | Qué separa | Proporción |
|---|---|---|
| Externa | Entrenamiento total / **Test** | 80 % / 20 % |
| Interna | Entrenamiento / **Validación** | 80 % / 20 % del entrenamiento total |

Resultado final sobre 150 muestras: **96 entrenamiento (64 %), 24 validación (16 %), 30 test (20 %)**.

## Fórmula de evaluación: exactitud (accuracy)

$$\text{accuracy} = \frac{\text{predicciones correctas}}{\text{total de predicciones}} = \frac{1}{n}\sum_{i=1}^{n}\mathbb{1}\left[y_i = \hat{y}_i\right]$$

Error asociado: $E = 1 - \text{accuracy}$

## Fórmula de estandarización (StandardScaler)

Se usa para llevar cada atributo a media 0 y desviación estándar 1:

$$z = \frac{x - \mu}{\sigma}$$

Donde:
- $x$ = valor original
- $\mu$ = media de la columna (calculada **solo con entrenamiento**)
- $\sigma$ = desviación estándar de la columna
- $z$ = valor estandarizado

## Celda 1: Importación de librerías

| Librería | Para qué se usa |
|---|---|
| `numpy` | Contar clases con `np.unique` y redondear resultados |
| `datasets` | Cargar el dataset Iris incluido en scikit-learn |
| `DummyClassifier` | Modelo "tonto" de referencia (baseline) |
| `train_test_split` | Hacer la partición Hold Out |
| `StandardScaler` | Estandarizar atributos con la fórmula z |

In [1]:
import numpy as np
from sklearn import datasets
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Celda 2: Carga del dataset

`datasets.load_iris()` devuelve un objeto con:
- `iris.data`: matriz de 150 × 4 (largo y ancho de sépalo y pétalo)
- `iris.target`: etiqueta de cada flor (0, 1, 2)
- `iris.target_names`: nombre de cada clase

Imprimimos los nombres de las 3 especies.

In [2]:
# Carga de datos
iris = datasets.load_iris()
print(iris.target_names)

['setosa' 'versicolor' 'virginica']


## Celda 3: Características del dataset

- `iris.data.shape` da (filas, columnas) = **(150, 4)**.
- `set(iris.target)` muestra las clases únicas.
- `np.unique(..., return_counts=True)` cuenta cuántas muestras hay por clase.

El dataset está **balanceado**: 50 muestras por clase.

In [3]:
# Mostrar características de la tabla de datos.
print("Tabla de datos: %d instancias y %d atributos" % (iris.data.shape[0], iris.data.shape[1]))
print("Valores de la clase:", set(iris.target))

# Cuantificamos el número de instancias que contiene el dataset por clase
valores, ocurrencias = np.unique(iris.target, return_counts=True)
print(valores, ocurrencias)

Tabla de datos: 150 instancias y 4 atributos
Valores de la clase: {np.int64(0), np.int64(1), np.int64(2)}
[0 1 2] [50 50 50]


## Celda 4: Partición externa (Entrenamiento / Test)

`train_test_split` separa los datos de forma aleatoria:
- `test_size=0.2`: 20 % para test (30 muestras), 80 % para entrenamiento (120).
- `random_state=42`: fija la semilla para que la división **sea siempre la misma** (reproducible).

Luego se cuenta cuántas muestras de cada clase cayeron en test. Como no se usó `stratify`, la distribución no es perfectamente igual (10, 9, 11).

> El conjunto de test se guarda "bajo llave": solo se usa al final para estimar el **error de generalización**.

In [4]:
# Test: hold-out split 80-20%. # Partición externa
X_training, X_test, y_training, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)
valores_test, ocur_test = np.unique(y_test, return_counts=True)
print('Test: ', 'clases:', valores_test, ' ocurrencias: ', ocur_test)

Test:  clases: [0 1 2]  ocurrencias:  [10  9 11]


## Celda 5: Estandarización

1. `fit_transform(X_training)`: **calcula** $\mu$ y $\sigma$ con los datos de entrenamiento y los transforma.
2. `transform(X_test)`: transforma el test usando la **misma** $\mu$ y $\sigma$ del entrenamiento (nunca se recalculan con test).

Así se evita que información del test se filtre al modelo (*data leakage*).

> **Nota:** aquí el escalador se ajusta antes de separar la validación, por lo que la validación influye levemente en $\mu$ y $\sigma$. Lo más estricto es ajustar el escalador solo con `X_train` (por ejemplo, con un `Pipeline`).

In [5]:
# Estandarizar las características de entrenamiento y de test
standardizer = StandardScaler()
X_training = standardizer.fit_transform(X_training)
X_test = standardizer.transform(X_test)

## Celda 6: Partición interna (Entrenamiento / Validación)

Se vuelve a aplicar Hold Out sobre el 80 % de entrenamiento:
- 80 % → `X_train`, `y_train` (96 muestras) para **ajustar** el modelo.
- 20 % → `X_val`, `y_val` (24 muestras) para **comparar modelos / ajustar hiperparámetros**.

| Conjunto | Función |
|---|---|
| Train | Aprender los parámetros |
| Validación | Elegir el mejor modelo |
| Test | Medir el rendimiento final, una sola vez |

In [6]:
# Validación: hold-out split 80-20%. # Partición interna
X_train, X_val, y_train, y_val = train_test_split(X_training, y_training, test_size=0.2, random_state=42)
valores_train, ocur_train = np.unique(y_train, return_counts=True)
print('Entrenamiento: ', ' clases:', valores_train, '  ocurrencias:', ocur_train)

valores_val, ocur_val = np.unique(y_val, return_counts=True)
print('Validation:    ', ' clases:', valores_val, '  ocurrencias:', ocur_val)

Entrenamiento:   clases: [0 1 2]   ocurrencias: [32 30 34]
Validation:      clases: [0 1 2]   ocurrencias: [ 8 11  5]


## Celda 7: Creación del modelo base (DummyClassifier)

`DummyClassifier(strategy='prior')` **ignora los atributos** y siempre predice la clase más frecuente del entrenamiento. Sirve como **línea base**: cualquier modelo real debe superarlo.

(En el notebook original esta celda estaba duplicada; se dejó una sola.)

In [7]:
# Construcción del objeto que contiene el algoritmo de aprendizaje.
clf = DummyClassifier(strategy='prior', random_state=42)

## Celda 8: Entrenamiento

`fit(X_train, y_train)` aprende los datos de entrenamiento. En un `DummyClassifier` solo calcula la frecuencia de cada clase (la mayoritaria es la 2, con 34 muestras).

In [8]:
# Entrenamiento del algoritmo de aprendizaje.
clf = clf.fit(X_train, y_train)

## Celda 9: Evaluación con `score`

`clf.score(X, y)` predice sobre `X` y devuelve la **exactitud**:

$$\text{accuracy} = \frac{\text{aciertos}}{\text{total}}$$

- Validación: 5/24 = 0.2083
- Test: 11/30 = 0.3667

Son bajas porque el modelo siempre responde la misma clase. Estas cifras son la referencia mínima a superar.

In [9]:
# Evaluación del algoritmo de aprendizaje con el método "score" que devuelve directamente la métrica de 'accuracy'
val_accuracy = clf.score(X_val, y_val)
print("Exactitud en validación: ", val_accuracy)

test_accuracy = clf.score(X_test, y_test)
print("Exactitud en test: ", test_accuracy)

Exactitud en validación:  0.20833333333333334
Exactitud en test:  0.36666666666666664


## Celda 10: Predicciones vs. etiquetas reales

`predict` devuelve la clase predicha para cada muestra. Se observa que el `DummyClassifier` predice **siempre 2**, sin importar la flor.

**Corrección:** en el original, la parte de test imprimía `y_pred_val` y `y_val` (variables de validación). Ahora imprime `y_pred_test` y `y_test`.

In [10]:
# Obtenemos las predicciones sobre conjunto de validación y de test
y_pred_val = clf.predict(X_val)
print('Predicciones de validación ', y_pred_val)
print('Etiquetas reales validación', y_val)

y_pred_test = clf.predict(X_test)
print('\nPredicciones de test ', y_pred_test)
print('Etiquetas reales test', y_test)

Predicciones de validación  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales validación [1 1 0 0 0 2 1 2 2 2 1 1 1 1 1 0 2 0 1 0 1 1 0 0]

Predicciones de test  [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
Etiquetas reales test [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


## Celda 11: Clasificador más complejo (SVC)

`SVC` (Support Vector Classifier) busca la frontera que mejor separa las clases.
- `C=0.5`: hiperparámetro de regularización. Un `C` pequeño tolera más errores y da una frontera más suave; un `C` grande se ajusta más a los datos.

Se entrena con `fit`, se evalúa en validación y en test, y se convierte la exactitud a porcentaje con `np.round(x*100, 4)`.

Resultado: **91.67 % en validación y 96.67 % en test**, muy superior al baseline (20.83 % y 36.67 %). Como val y test son similares, el modelo **generaliza bien**.

In [11]:
# Aplicamos un ejemplo con un clasificador más complejo que el "dummyclassifier"
from sklearn.svm import SVC
svc = SVC(C=0.5) # Definimos algoritmo
svc.fit(X_train, y_train) # Entrenamos modelo

val_accuracy = svc.score(X_val, y_val) # Evaluamos modelo en validación
print('Exactitud en validación ', np.round(val_accuracy*100, 4), '%')

test_accuracy = svc.score(X_test, y_test) # Evaluamos modelo en test
print('Exactitud en test ', np.round(test_accuracy*100, 4), '%')

Exactitud en validación  91.6667 %
Exactitud en test  96.6667 %


## Celda 12: Guardar y cargar el modelo (pickle)

`pickle` serializa el objeto entrenado a un archivo para reutilizarlo sin volver a entrenar.
- `'wb'`: escribir en binario → `pickle.dump(objeto, archivo)`.
- `'rb'`: leer en binario → `pickle.load(archivo)`.

**Corrección del error original:** daba `NameError: name 'model' is not defined` porque la variable `model` no existía; el modelo entrenado se llama `svc`. Además se agrega `os.makedirs` para crear la carpeta `../models` si no existe.

In [12]:
import os
import pickle

os.makedirs('../models', exist_ok=True)

# Guardar modelo
with open('../models/model.pickle', 'wb') as fw:
    pickle.dump(svc, fw)

# Cargar modelo
with open('../models/model.pickle', 'rb') as fr:
    modelo_cargado = pickle.load(fr)

# Verificación: el modelo cargado debe dar la misma exactitud
print('Exactitud del modelo cargado en test:', np.round(modelo_cargado.score(X_test, y_test)*100, 4), '%')

Exactitud del modelo cargado en test: 96.6667 %


## Conclusiones

- Hold Out separa los datos en entrenamiento, validación y test para estimar cómo se comportará el modelo con datos nuevos.
- El `DummyClassifier` (baseline) obtuvo 20.83 % en validación y 36.67 % en test.
- El `SVC` obtuvo 91.67 % y 96.67 %, por lo que aprende patrones reales del dataset.
- Limitación: con solo 24 muestras de validación y 30 de test, la estimación varía según la semilla. Para resultados más estables se puede usar `stratify=y` o validación cruzada.